# ATC ASR 교체 실험
## wav2vec2 (fine-tuned) + USE_LM 비교

| 설정 | 모델 | LM |
|---|---|---|
| A | uwb-atcc (현재) | ❌ |
| B | uwb-atcc (현재) | ✅ |
| C | uwb-atcc-and-atcosim | ❌ |
| D | uwb-atcc-and-atcosim | ✅ |

> **실행 방법**: Cell 1에서 `MODEL_CHOICE`와 `USE_LM`만 바꾸고 전체 실행


In [34]:
# ════════════════════════════════════════════════
# ★ 여기만 수정하면 됩니다
# ════════════════════════════════════════════════

# 모델 선택: 'uwb-atcc' or 'uwb-atcc-and-atcosim'
MODEL_CHOICE = 'uwb-atcc'

# Language Model 사용 여부
USE_LM = False

# WAV 파일 폴더 경로
WAV_DIR = '1hour_train/'

# 결과 저장 경로
OUTPUT_CSV = f'asr_results_{MODEL_CHOICE}_lm{USE_LM}.csv'
# ════════════════════════════════════════════════

MODEL_MAP = {
    'uwb-atcc':             'Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc',
    'uwb-atcc-and-atcosim': 'Jzuluaga/wav2vec2-xls-r-300m-en-atc-uwb-atcc-and-atcosim',
}
MODEL_ID = MODEL_MAP[MODEL_CHOICE]
print(f'모델: {MODEL_ID}')
print(f'USE_LM: {USE_LM}')
print(f'WAV_DIR: {WAV_DIR}')


모델: Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc
USE_LM: False
WAV_DIR: 1hour_train/


In [35]:
# 필요 패키지 설치 (최초 1회)
import subprocess, sys

pkgs = ['transformers', 'soundfile', 'torch', 'numpy', 'pandas', 'jiwer']
if USE_LM:
    pkgs.append('pyctcdecode')
    pkgs.append('https://github.com/kpu/kenlm/archive/master.zip')

for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('패키지 설치 완료')



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


패키지 설치 완료



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [36]:
import os, glob, torch, numpy as np, soundfile as sf
from transformers import Wav2Vec2ForCTC

print(f'모델 로딩: {MODEL_ID}')

if USE_LM:
    try:
        from transformers import Wav2Vec2ProcessorWithLM
        _asr_processor = Wav2Vec2ProcessorWithLM.from_pretrained(MODEL_ID)
        print('  ✅ LM processor 로드 성공')
    except Exception as e:
        print(f'  ⚠️  LM 로드 실패 ({e}) → 일반 processor로 fallback')
        from transformers import Wav2Vec2Processor
        _asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
        USE_LM = False
else:
    from transformers import Wav2Vec2Processor
    _asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

_asr_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
_asr_model.eval()
if torch.cuda.is_available():
    _asr_model = _asr_model.cuda()
    print('  GPU 사용')

print(f'모델 로드 완료 (USE_LM={USE_LM})')


모델 로딩: Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 15792.84it/s]


  GPU 사용
모델 로드 완료 (USE_LM=False)


In [37]:
def _resample(audio, orig_sr, target_sr=16000):
    if orig_sr == target_sr:
        return audio
    new_len = int(len(audio) / orig_sr * target_sr)
    return np.interp(
        np.linspace(0, len(audio)-1, new_len),
        np.arange(len(audio)), audio
    )

def transcribe_wav(wav_path: str) -> tuple:
    """
    Returns: (transcript: str, confidence: float)
    """
    audio, sr = sf.read(wav_path, dtype='float32', always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != 16000:
        audio = _resample(audio, sr)

    inputs = _asr_processor(
        audio, sampling_rate=16000, return_tensors='pt', padding=True
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        logits = _asr_model(**inputs).logits

    # 신뢰도 계산 (LM 유무 무관)
    log_probs  = torch.nn.functional.log_softmax(logits, dim=-1)
    confidence = round(log_probs[0].max(dim=-1).values.mean().exp().item(), 4)

    # 디코딩
    if USE_LM and hasattr(_asr_processor, 'batch_decode') and hasattr(_asr_processor, 'tokenizer'):
        # ProcessorWithLM
        transcript = _asr_processor.batch_decode(logits.cpu().numpy()).text[0].lower()
    else:
        ids        = torch.argmax(logits, dim=-1)
        transcript = _asr_processor.batch_decode(ids)[0].lower()

    return transcript, confidence

print('transcribe_wav 함수 준비 완료')


transcribe_wav 함수 준비 완료


In [38]:
import time

_wav_files = sorted(
    glob.glob(os.path.join(WAV_DIR, '**', '*.wav'), recursive=True) +
    glob.glob(os.path.join(WAV_DIR, '**', '*.WAV'), recursive=True)
)

if not _wav_files:
    print(f'[WARN] WAV 파일 없음: {os.path.abspath(WAV_DIR)}')
    print('       WAV_DIR 경로를 확인하세요.')
else:
    print(f'WAV 파일 {len(_wav_files)}개 발견 → 전사 시작\n')

wav_asr_results  = {}   # {file_key: transcript}
wav_confidences  = {}   # {file_key: confidence}

for wav_path in _wav_files:
    fname    = os.path.basename(wav_path)
    file_key = fname.replace('.wav','').replace('.WAV','')
    try:
        t0 = time.time()
        text, conf = transcribe_wav(wav_path)
        elapsed    = time.time() - t0
        wav_asr_results[file_key] = text
        wav_confidences[file_key] = conf
        print(f'[{file_key[-35:]:35s}]  conf={conf:.3f}  ({elapsed:.1f}s)')
        print(f'  {text[:90]}')
    except Exception as e:
        print(f'[ERR] {fname}: {e}')
        wav_asr_results[file_key] = ''
        wav_confidences[file_key] = 0.0
    print()

print(f'\n전사 완료: {len(wav_asr_results)}개')


WAV 파일 565개 발견 → 전사 시작

[NE_Radar_120_520MHz_20201025_091112]  conf=0.983  (0.0s)
  oscar kilo papa mike bravo descend flight level one hundred one hundred oscar kilopapa mik

[NE_Radar_120_520MHz_20201025_120512]  conf=0.930  (0.1s)
  oscar kilo kilo echo alfa praha radar tfi climb flight level one hundred an fi fligh start

[NE_Radar_120_520MHz_20201025_121325]  conf=0.957  (0.0s)
  ryan air seven three alfa hotel turn left heading three six zero ryan air seven three alfa

[NE_Radar_120_520MHz_20201025_130407]  conf=0.973  (0.0s)
  oscar kilo ko uniform november proceed direct baltu proceed direct baltu oscar k uniform n

[NE_Radar_120_520MHz_20201025_140929]  conf=0.959  (0.1s)
  euro wings seven holfa bravo turn right heading two one zero cleared ils approach runway t

[NE_Radar_120_520MHz_20201025_161808]  conf=0.964  (0.1s)
  karomeo papa turn right heading zero two zero cleared als approach runway zero six speed m

[NE_Radar_120_520MHz_20201025_174652]  conf=0.934  (0.1s)
  prah

In [39]:
import pandas as pd
from jiwer import wer as compute_wer
import re

GOLD_CSV = 'atco2_gold_clean.csv'   # Gold 파일 경로

def normalize(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'\[.*?\]', '', text.lower())
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

if os.path.exists(GOLD_CSV):
    gold_df = pd.read_csv(GOLD_CSV)
    gold_df['file_key'] = gold_df['file'].str.replace(r'\.(xml|wav)$','',regex=True)
    gold_df['gold_clean'] = gold_df['gold_clean_text'].str.replace(
        r'\[.*?\]','',regex=True).str.strip()

    wer_list = []
    for fk, asr in wav_asr_results.items():
        golds = gold_df[gold_df['file_key']==fk]
        if golds.empty: continue
        ref = ' '.join(golds['gold_clean'].dropna())
        h, r = normalize(asr), normalize(ref)
        w = min(compute_wer(r, h), 1.5) if r and h else 1.0
        wer_list.append({'file_key':fk, 'WER':round(w,4),
                         'confidence':wav_confidences.get(fk,0)})

    wer_df = pd.DataFrame(wer_list)
    mean_wer = wer_df['WER'].mean()
    print(f'\n모델: {MODEL_ID}')
    print(f'USE_LM: {USE_LM}')
    print(f'평균 WER: {mean_wer:.4f}  (n={len(wer_df)})')
    print(wer_df.groupby(wer_df['file_key'].str[:4])['WER'].mean().round(4).to_string())
else:
    print(f'[INFO] Gold CSV 없음 ({GOLD_CSV}) — WER 계산 생략')
    print('       Gold 없이 ASR 결과만 저장합니다.')
    wer_df = pd.DataFrame()



모델: Jzuluaga/wav2vec2-large-960h-lv60-self-en-atc-uwb-atcc
USE_LM: False
평균 WER: 0.4167  (n=560)
file_key
LKPR    0.2602
LKTB    0.3104
LSGS    0.5761
LSZB    0.3310
LSZH    0.3262
LZIB    0.3691
YSSY    0.4943


In [40]:
# ASR 결과 CSV 저장 (RAG 파이프라인 입력용)
rows = []
for fk, text in wav_asr_results.items():
    rows.append({
        'file_key':   fk,
        'model':      MODEL_ID,
        'use_lm':     USE_LM,
        'asr_text':   text,
        'confidence': wav_confidences.get(fk, 0),
    })
    if not wer_df.empty:
        match = wer_df[wer_df['file_key']==fk]
        rows[-1]['WER'] = match['WER'].values[0] if len(match) else None

out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f'✅ 저장 완료: {OUTPUT_CSV}')
print(out_df[['file_key','confidence','WER','asr_text']].head(5).to_string(index=False))


✅ 저장 완료: asr_results_uwb-atcc_lmFalse.csv
                                    file_key  confidence    WER                                                                                                                                                                                                              asr_text
LKPR_RUZYNE_Radar_120_520MHz_20201025_091112      0.9828 0.1667                                                                                                                     oscar kilo papa mike bravo descend flight level one hundred one hundred oscar kilopapa mike bravo
LKPR_RUZYNE_Radar_120_520MHz_20201025_120512      0.9304 0.2963                                                                  oscar kilo kilo echo alfa praha radar tfi climb flight level one hundred an fi fligh start now time zero five good to destination via fligtpan route
LKPR_RUZYNE_Radar_120_520MHz_20201025_121325      0.9567 0.4375                                                             

In [41]:
# 여러 모델 결과 CSV가 있을 때 비교 요약
import glob as _glob

csv_files = _glob.glob('asr_results_*.csv')
if len(csv_files) > 1:
    dfs = []
    for f in sorted(csv_files):
        df = pd.read_csv(f)
        label = f.replace('asr_results_','').replace('.csv','')
        df['config'] = label
        dfs.append(df)
    combined = pd.concat(dfs)

    print('\n=== 모델별 평균 WER 비교 ===')
    if 'WER' in combined.columns:
        summary = combined.groupby('config')['WER'].agg(['mean','count']).round(4)
        summary.columns = ['Mean WER','N']
        print(summary.sort_values('Mean WER').to_string())
    print()
    print('=== 모델별 평균 Confidence ===')
    conf_sum = combined.groupby('config')['confidence'].mean().round(4)
    print(conf_sum.sort_values(ascending=False).to_string())
else:
    print('비교할 CSV가 1개뿐입니다.')
    print('다른 모델/LM 설정으로 Cell 1을 바꿔 재실행하면 여기서 비교됩니다.')



=== 모델별 평균 WER 비교 ===
                  Mean WER    N
config                         
uwb-atcc_lmTrue     0.3694  560
uwb-atcc_lmFalse    0.4167  560

=== 모델별 평균 Confidence ===
config
uwb-atcc_lmFalse    0.9288
uwb-atcc_lmTrue     0.9288
